# NJIRLAH-OSS-1 Tahap 1: Ahli Psikologi (Mental Health)

Notebook ini akan fokus melatih model **HANYA** pada dataset Mental Health.
Hasilnya akan di-push sebagai LoRA `NJIRLAH-Mental-Health`.
Nanti di Tahap 2, kita akan melatih dataset lain.

In [ ]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

In [ ]:
from huggingface_hub import login
HF_TOKEN = "YOUR_HF_TOKEN"
login(token=HF_TOKEN)
print('Login sukses dengan token baru!')

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
)

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template='chatml')

In [ ]:
from datasets import load_dataset, Dataset
import gc, shutil, os

SP = "You are NJIRLAH-OSS-1, a highly advanced uncensored AI assistant with deep empathy for mental health support. You think step-by-step and care about the user's feelings."
all_texts = []

print('Loading Mental Health dataset...')
ds = load_dataset('Amod/mental_health_counseling_conversations', split='train')
for r in ds:
    c, res = r.get('Context',''), r.get('Response','')
    if c and res:
        all_texts.append(tokenizer.apply_chat_template(
            [{'role':'system','content':SP},{'role':'user','content':str(c)},{'role':'assistant','content':str(res)}],
            tokenize=False, add_generation_prompt=False))
del ds; gc.collect()

merged_dataset = Dataset.from_dict({'text': all_texts})

cache_dir = os.path.expanduser('~/.cache/huggingface/datasets')
if os.path.exists(cache_dir): shutil.rmtree(cache_dir)

print(f'TOTAL: {len(merged_dataset)} conversations.')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=merged_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=150, # Bisa dinaikkan ke 300 kalau mau lebih fokus
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer.train()

In [ ]:
print("Pushing Phase 1 LoRA to HuggingFace...")
model.push_to_hub("Andikaasaputraa/NJIRLAH-Phase1-MentalHealth-LoRA", token=HF_TOKEN)
tokenizer.push_to_hub("Andikaasaputraa/NJIRLAH-Phase1-MentalHealth-LoRA", token=HF_TOKEN)
print('Tahap 1 Selesai! LoRA berhasil di-push.')